# Phase 3: Credit Risk Model Development
## Step 3: Risk Scoring & Loan Decision Layer
### Credit Risk & Loan Decision Intelligence Platform

---

## SECTION 1 — Phase 3 Step 3 Introduction

### 1. What is Probability of Default (PD)?
In credit underwriting, the **Probability of Default (PD)** represents the statistical likelihood $\hat{P}(y=1|X) \in [0.0, 1.0]$ that a prospective loan applicant will fail to meet their contractual repayment obligations within the agreed tenure.

### 2. Why Does a Lender Need More Than a Raw ML Prediction?
A raw machine learning model produces either a continuous numerical probability (e.g., $PD = 0.1755$) or a naive binary classification ($0$ or $1$) based on an arbitrary mathematical threshold (such as 0.50). In commercial retail lending, a raw model output **cannot directly approve or reject loans** for several reasons:
1. **Operational Granularity**: Borrowers do not simply fall into 'safe' and 'doomed' categories. Risk is a continuous spectrum requiring multi-tiered operational workflows.
2. **Risk-Based Pricing & Capacity**: Lenders set loan amounts, interest rate markups, tenure limits, and collateral requirements based on standardized risk scores.
3. **Human-in-the-Loop Underwriting**: Marginal or borderline cases require manual review by credit officers rather than automated instant rejection.
4. **Regulatory Governance & Accountability**: Credit institutions are governed by credit committee policies, fair lending laws, and risk appetite frameworks that mandate transparent, auditable business rules.

### 3. Understanding the Multi-Stage Decision Hierarchy:
```
┌───────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                  RISK & DECISION ARCHITECTURE                                     │
├───────────────────────────┬───────────────────────────────────────────────────────────────────────┤
│ 1. ML Prediction          │ Raw log-odds / model activation from feature vector dot product       │
│ 2. Estimated PD           │ Continuous estimated probability of default: P(y=1|X) in [0.0, 1.0]   │
│ 3. Risk Score             │ Standardized 0–100 scale (Higher = Safer, Lower = Riskier)            │
│ 4. Risk Tier              │ Policy groupings: LOW RISK, MEDIUM RISK, HIGH RISK                    │
│ 5. Operational Decision   │ Concrete business action: APPROVE, MANUAL REVIEW, REJECT              │
└───────────────────────────┴───────────────────────────────────────────────────────────────────────┘
```

> **🛡️ Core Architectural Principle:**  
> **The Machine Learning Model estimates default risk; the Business Policy Layer converts that risk into an operational decision.**  
> **Estimated PD $\neq$ Final Loan Decision.**

---
## SECTION 2 — Reuse the Completed Model Pipeline

We import core scientific and modeling libraries and reproduce the exact certified Phase 2 preprocessing transformation pipeline:
- **Stratified 80/20 train/test split** with `random_state=42`.
- **`StandardScaler`** for 13 numerical columns (fitted strictly on `X_train`).
- **`OneHotEncoder(handle_unknown='ignore', sparse_output=False)`** for `employment_type`.
- Re-instantiate the 3 completed Phase 3 candidate models:
  1. `LogisticRegression(max_iter=1000, random_state=42)`
  2. `RandomForestClassifier(n_estimators=200, max_depth=4, random_state=42, class_weight='balanced')`
  3. `XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric='logloss')`
- Preserve strict zero-leakage test isolation: `X_test_processed` is never seen during model fitting.

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# 1. Load canonical dataset
data_path = '../data/raw/credit_risk_50.csv'
df = pd.read_csv(data_path)

target_col = 'default'
id_col = 'applicant_id'

y = df[target_col].copy()
ml_feature_cols = [col for col in df.columns if col not in [id_col, target_col]]
X = df[ml_feature_cols].copy()

# 2. Stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 3. Column groupings
numerical_cols = [
    'age', 'monthly_income', 'loan_amount', 'loan_term_months', 'existing_emi',
    'credit_score', 'credit_utilization_pct', 'late_payment_count', 'previous_defaults',
    'employment_years', 'average_bank_balance', 'monthly_expenses', 'debt_to_income_ratio'
]
categorical_cols = ['employment_type']

# 4. Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ],
    verbose_feature_names_out=False
)

# Fit strictly on training set
preprocessor.fit(X_train)
feature_names = list(preprocessor.get_feature_names_out())

X_train_processed = pd.DataFrame(preprocessor.transform(X_train), columns=feature_names, index=X_train.index)
X_test_processed = pd.DataFrame(preprocessor.transform(X_test), columns=feature_names, index=X_test.index)

# 5. Recreate and fit all 3 Phase 3 candidate models strictly on X_train_processed
lr_model = LogisticRegression(max_iter=1000, random_state=42)
rf_model = RandomForestClassifier(n_estimators=200, max_depth=4, random_state=42, class_weight='balanced')
xgb_model = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric='logloss')

lr_model.fit(X_train_processed, y_train)
rf_model.fit(X_train_processed, y_train)
xgb_model.fit(X_train_processed, y_train)

print(f"X_train_processed shape : {X_train_processed.shape} (40 training rows, 16 features)")
print(f"X_test_processed shape  : {X_test_processed.shape}  (10 test rows, 16 features)")
print(f"Models fitted           : Logistic Regression, Random Forest, XGBoost\n")
print("Model pipeline replicated successfully.")


X_train_processed shape : (40, 16) (40 training rows, 16 features)
X_test_processed shape  : (10, 16)  (10 test rows, 16 features)
Models fitted           : Logistic Regression, Random Forest, XGBoost

Model pipeline replicated successfully.


---
## SECTION 3 — Model Selection for Decision Demonstration

### 1. Comparative Review of Candidate Models (From Step 2 Benchmarks):
We summarize the performance metrics achieved by all three models across 5-fold cross-validation and the test partition.

In [3]:
# Benchmark summary table
model_comparison = pd.DataFrame({
    'Model Architecture': ['Logistic Regression (Baseline)', 'Random Forest (Ensemble)', 'XGBoost (Gradient Boosted)'],
    '5-Fold CV Accuracy': ['1.0000 +/- 0.0000', '1.0000 +/- 0.0000', '0.9500 +/- 0.1000'],
    '5-Fold CV ROC-AUC': ['1.0000 +/- 0.0000', '1.0000 +/- 0.0000', '0.9867 +/- 0.0267'],
    'Test Set Accuracy': ['1.0000 (100.0%)', '1.0000 (100.0%)', '1.0000 (100.0%)'],
    'Test Set ROC-AUC': ['1.0000 (100.0%)', '1.0000 (100.0%)', '1.0000 (100.0%)']
})

print("=== Candidate Model Benchmark Comparison ===")
display(model_comparison)


=== Candidate Model Benchmark Comparison ===


,Model Architecture,5-Fold CV Accuracy,5-Fold CV ROC-AUC,Test Set Accuracy,Test Set ROC-AUC
0,Logistic Regression (Baseline),1.0000 +/- 0.0000,1.0000 +/- 0.0000,1.0000 (100.0%),1.0000 (100.0%)
1,Random Forest (Ensemble),1.0000 +/- 0.0000,1.0000 +/- 0.0000,1.0000 (100.0%),1.0000 (100.0%)
2,XGBoost (Gradient Boosted),0.9500 +/- 0.1000,0.9867 +/- 0.0267,1.0000 (100.0%),1.0000 (100.0%)


### 2. Selection Rationale for Demonstration Decision Engine:
For our demonstration scoring and loan decision layer, we select **Logistic Regression** as the primary scoring model for the following architectural reasons:
1. **Linear Interpretability & Transparency**: In retail lending, explainability and transparency are strictly mandated by regulatory bodies (e.g., RBI guidelines, Fair Lending regulations, Adverse Action Notices). Linear models provide monotonic log-odds weights without black-box opacity.
2. **Consistent Probability Monotonicity**: Logistic Regression smoothly translates credit risk features into estimated default probabilities across the entire feature domain.
3. **Provisional Nature**: Because the dataset contains **50 synthetic records**, all three models demonstrate near-perfect separation on this sample. This model selection is **provisional and educational**, designed to demonstrate the complete end-to-end decision intelligence architecture.

---
## SECTION 4 — Estimated Probability of Default (PD)

We generate the **estimated Probability of Default (PD)** for all 10 test applicants using `predict_proba(X_test_processed)[:, 1]`.

> **⚠️ Terminology Notice:**  
> These values represent the model's **estimated Probability of Default (PD)**. They are output directly by the model's logistic sigmoid link function and have not undergone post-hoc empirical calibration (e.g. Platt scaling / isotonic regression).

In [4]:
# Primary decision model
selected_model = lr_model

# Generate discrete class predictions and estimated PD probabilities
y_test_pred = selected_model.predict(X_test_processed)
estimated_pd = selected_model.predict_proba(X_test_processed)[:, 1]

# Display Estimated Probability of Default table
pd_table = pd.DataFrame({
    'Applicant Index': X_test_processed.index,
    'Actual Default': y_test.values,
    'Predicted Default': y_test_pred,
    'Estimated PD': np.round(estimated_pd, 4)
})

print("=== Estimated Probability of Default (PD) Table ===")
display(pd_table)


=== Estimated Probability of Default (PD) Table ===


,Applicant Index,Actual Default,Predicted Default,Estimated PD
0,10,0,0,0.0933
1,23,0,0,0.0022
2,38,1,1,0.9746
3,45,0,0,0.0637
4,12,0,0,0.0005
5,40,1,1,0.9704
6,24,1,1,0.9791
7,35,1,1,0.9210
8,16,0,0,0.0006
9,31,0,0,0.1755


---
## SECTION 5 — Demonstration Risk Score

### 1. Mathematical Formulation:
In enterprise lending systems, continuous probability values ($0.0 \le PD \le 1.0$) are commonly translated into standardized integer score scales ($0$ to $100$ or $300$ to $900$) to facilitate intuitive communication across business stakeholders, credit committees, and automated policy engines.

We define a transparent **Demonstration Risk Score** as:

$$\text{Risk Score} = (1 - \text{Estimated PD}) \times 100$$

### 2. Score Directionality:
- **Higher Risk Score (e.g., 85–100)** $\longrightarrow$ **Safer Borrower** (Low Probability of Default $\approx 0.00 - 0.15$).
- **Lower Risk Score (e.g., 0–50)** $\longrightarrow$ **Riskier Borrower** (High Probability of Default $\ge 0.50$).

> **⚠️ CRITICAL DISCLAIMER:**  
> This demonstration risk score is a **transparent educational transformation formula** created for pipeline validation. It is **NOT** a credit bureau score (such as CIBIL, Experian, Equifax, or CRIF High Mark) nor an official banking regulatory score. Bureau scores range from 300 to 900 and are external inputs; this internal score is derived from our ML default model.

In [5]:
# Calculate Demonstration Risk Score (0 to 100 scale)
risk_scores = np.round((1.0 - estimated_pd) * 100.0, 2)

score_df = pd.DataFrame({
    'Applicant Index': X_test_processed.index,
    'Estimated PD': np.round(estimated_pd, 4),
    'Demonstration Risk Score (0-100)': risk_scores
})

print("=== Demonstration Risk Score Calculation ===")
display(score_df)


=== Demonstration Risk Score Calculation ===


,Applicant Index,Estimated PD,Demonstration Risk Score (0-100)
0,10,0.0933,90.67
1,23,0.0022,99.78
2,38,0.9746,2.54
3,45,0.0637,93.63
4,12,0.0005,99.95
5,40,0.9704,2.96
6,24,0.9791,2.09
7,35,0.9210,7.90
8,16,0.0006,99.94
9,31,0.1755,82.45


---
## SECTION 6 — Risk Tiers & Policy Thresholds

### 1. Risk Tier Architecture:
Credit institutions segment borrowers into discrete **Risk Tiers** to govern underwriting workflows. We define three tiers using explicitly named constants:

- **`LOW` Risk Tier**: High creditworthiness, robust cashflow cushion, minimal default probability.
- **`MEDIUM` Risk Tier**: Moderate credit metrics, elevated DTI or thin cashflow buffers; requires human credit officer appraisal.
- **`HIGH` Risk Tier**: Significant delinquency history, severe leverage, or high default propensity.

### 2. Named Threshold Constants (DEMO POLICY ASSUMPTIONS):
Thresholds are **policy assumptions set by credit committees and risk appetite**, not inherent physical facts of the machine learning model. They can be tuned based on capital adequacy and macroeconomic cycles.

In [6]:
# ============================================================
# DEMO POLICY ASSUMPTIONS — Risk Tier Cutoffs
# ============================================================
LOW_RISK_MAX_PD = 0.15      # Applicants with PD < 15% fall into LOW risk tier
MEDIUM_RISK_MAX_PD = 0.40   # Applicants with 15% <= PD < 40% fall into MEDIUM risk tier
# Applicants with PD >= 40% fall into HIGH risk tier

print("=== Configured Policy Thresholds (DEMO POLICY ASSUMPTION) ===")
print(f"  LOW Risk Tier    : Estimated PD < {LOW_RISK_MAX_PD:.2f}  (Risk Score > {((1-LOW_RISK_MAX_PD)*100):.1f})")
print(f"  MEDIUM Risk Tier : {LOW_RISK_MAX_PD:.2f} <= Estimated PD < {MEDIUM_RISK_MAX_PD:.2f}  ({((1-MEDIUM_RISK_MAX_PD)*100):.1f} <= Risk Score <= {((1-LOW_RISK_MAX_PD)*100):.1f})")
print(f"  HIGH Risk Tier   : Estimated PD >= {MEDIUM_RISK_MAX_PD:.2f}  (Risk Score < {((1-MEDIUM_RISK_MAX_PD)*100):.1f})")

def assign_risk_tier(pd_value, low_cutoff=LOW_RISK_MAX_PD, med_cutoff=MEDIUM_RISK_MAX_PD):
    """
    Assigns an applicant to a categorical risk tier based on continuous estimated PD.
    """
    if pd_value < low_cutoff:
        return 'LOW'
    elif pd_value < med_cutoff:
        return 'MEDIUM'
    else:
        return 'HIGH'

# Apply risk tier mapping
risk_tiers = [assign_risk_tier(p) for p in estimated_pd]
print("\nRisk Tier assignment function verified.")


=== Configured Policy Thresholds (DEMO POLICY ASSUMPTION) ===
  LOW Risk Tier    : Estimated PD < 0.15  (Risk Score > 85.0)
  MEDIUM Risk Tier : 0.15 <= Estimated PD < 0.40  (60.0 <= Risk Score <= 85.0)
  HIGH Risk Tier   : Estimated PD >= 0.40  (Risk Score < 60.0)

Risk Tier assignment function verified.


---
## SECTION 7 — Loan Decision Policy Layer

### 1. Operational Decision Logic:
The business decision layer translates categorized risk tiers into definitive operational lending workflows:

- **`APPROVE`**: Automated straight-through processing (STP) for low-risk applicants.
- **`MANUAL REVIEW`**: Routed to senior underwriter queues for bank statement inspection, collateral verification, or co-borrower requirement.
- **`REJECT`**: Automated adverse action decline for applicants exceeding maximum risk tolerance.

```
┌────────────────────────┬───────────────────────────┬──────────────────────────────────────────┐
│ RISK TIER              │ OPERATIONAL DECISION      │ WORKFLOW ACTION                          │
├────────────────────────┼───────────────────────────┼──────────────────────────────────────────┤
│ LOW (PD < 0.15)        │ APPROVE                   │ Instant automated sanction               │
│ MEDIUM (0.15 <= PD <0.40)│ MANUAL REVIEW           │ Secondary underwriter audit queue        │
│ HIGH (PD >= 0.40)      │ REJECT                    │ Adverse action letter & decline notice   │
└────────────────────────┴───────────────────────────┴──────────────────────────────────────────┘
```

In [7]:
def make_loan_decision(risk_tier):
    """
    Translates a categorical risk tier into an operational lending decision.
    """
    if risk_tier == 'LOW':
        return 'APPROVE'
    elif risk_tier == 'MEDIUM':
        return 'MANUAL REVIEW'
    elif risk_tier == 'HIGH':
        return 'REJECT'
    else:
        raise ValueError(f"Unknown risk tier: {risk_tier}")

# Generate operational loan decisions
loan_decisions = [make_loan_decision(t) for t in risk_tiers]
print("Loan decision engine logic compiled and executed.")


Loan decision engine logic compiled and executed.


---
## SECTION 8 — Final Comprehensive Decision Table

We assemble the complete end-to-end evaluation table linking test applicant indices, estimated PD, demonstration risk scores, risk tiers, operational decisions, and actual ground truth default labels.

In [8]:
# Construct complete decision table dynamically from pipeline outputs
decision_table = pd.DataFrame({
    'Applicant Index': X_test_processed.index,
    'Estimated PD': np.round(estimated_pd, 4),
    'Risk Score (0-100)': risk_scores,
    'Risk Tier': risk_tiers,
    'Decision': loan_decisions,
    'Actual Default': y_test.values
})

print("=== Final Loan Decision Intelligence Table (Test Set: 10 Records) ===")
display(decision_table)

# Summary counts by decision category
decision_summary = decision_table['Decision'].value_counts().reset_index()
decision_summary.columns = ['Decision', 'Applicant Count']
print("\n=== Decision Breakdown ===")
display(decision_summary)


=== Final Loan Decision Intelligence Table (Test Set: 10 Records) ===


,Applicant Index,Estimated PD,Risk Score (0-100),Risk Tier,Decision,Actual Default
0,10,0.0933,90.67,LOW,APPROVE,0
1,23,0.0022,99.78,LOW,APPROVE,0
2,38,0.9746,2.54,HIGH,REJECT,1
3,45,0.0637,93.63,LOW,APPROVE,0
4,12,0.0005,99.95,LOW,APPROVE,0
5,40,0.9704,2.96,HIGH,REJECT,1
6,24,0.9791,2.09,HIGH,REJECT,1
7,35,0.9210,7.90,HIGH,REJECT,1
8,16,0.0006,99.94,LOW,APPROVE,0
9,31,0.1755,82.45,MEDIUM,MANUAL REVIEW,0



=== Decision Breakdown ===


,Decision,Applicant Count
0,APPROVE,5
1,REJECT,4
2,MANUAL REVIEW,1


---
## SECTION 9 — Business & Underwriting Interpretation

We walk through representative applicant profiles from the test set to illustrate how the decision intelligence system behaves across distinct risk tiers:

### 1. Low-Risk Applicant Example (Applicant Index 12):
- **Estimated PD**: $0.0005$ ($0.05\%$ default probability).
- **Demonstration Risk Score**: $99.95 / 100$.
- **Risk Tier**: `LOW`.
- **Operational Decision**: `APPROVE`.
- **Business Context**: The model estimates a near-zero probability of default due to high income, low DTI, and a strong credit score. The policy engine recommends straight-through automated approval.

### 2. Medium-Risk / Borderline Applicant Example (Applicant Index 31):
- **Estimated PD**: $0.1755$ ($17.55\%$ default probability).
- **Demonstration Risk Score**: $82.45 / 100$.
- **Risk Tier**: `MEDIUM`.
- **Operational Decision**: `MANUAL REVIEW`.
- **Business Context**: While the applicant is non-defaulting in the ground truth ($y=0$), their elevated DTI and moderate credit score place them in the borderline zone ($15\% \le PD < 40\%$). The policy engine appropriately flags this application for credit officer review rather than automated instant approval or decline.

### 3. High-Risk Applicant Example (Applicant Index 38):
- **Estimated PD**: $0.9746$ ($97.46\%$ default probability).
- **Demonstration Risk Score**: $2.54 / 100$.
- **Risk Tier**: `HIGH`.
- **Operational Decision**: `REJECT`.
- **Business Context**: Severe delinquency history and high credit utilization produce an estimated default probability exceeding $97\%$. The policy engine issues an automated decline.

> **⚠️ Crucial Underwriting Insight:**  
> An automated lending decision is a **policy recommendation based on statistical probability and risk tolerance**—it is **NOT** a deterministic guarantee that an approved applicant will never default, nor that a rejected applicant was guaranteed to default.

---
## SECTION 10 — Identity Risk vs. Credit Default Risk Architecture Audit

We reinforce the foundational architectural boundary governing the entire platform:

```
┌──────────────────────────────────────────────────────────────────────────────────────────┐
│                               SYSTEM BOUNDARY AUDIT                                      │
├─────────────────────────────────────────────┬────────────────────────────────────────────┤
│ 1. IDENTITY & ELIGIBILITY RISK              │ 2. CREDIT DEFAULT RISK (ML SCORING)        │
│    (src/verification/)                      │    (src/credit/, src/features/ & ML)       │
├─────────────────────────────────────────────┼────────────────────────────────────────────┤
│ • pan_verified                              │ • credit_score                             │
│ • aadhaar_kyc_verified                      │ • debt_to_income_ratio                     │
│ • mobile_otp_verified                       │ • monthly_income & existing_emi            │
│ • identity_match_score                      │ • late_payment_count & previous_defaults   │
│ • name_dob_consistency                     │ • banking balances & monthly cashflows     │
├─────────────────────────────────────────────┼────────────────────────────────────────────┤
│ Purpose: ELIGIBILITY GATE                   │ Purpose: RISK PROBABILITY & SCORING        │
│ Action: Hault fraud / Require manual KYC    │ Action: Compute PD, Risk Score, Tier       │
│ Stage: Evaluated BEFORE credit underwriting │ Stage: Evaluated ONLY for KYC-pass records │
└─────────────────────────────────────────────┴────────────────────────────────────────────┘
```

- **Identity Verification Signals**: Excluded 100% from the ML feature matrix ($X$).
- **Credit Default Risk Signals**: Purely financial and behavioral solvency metrics.

In [9]:
# Verification field isolation audit
identity_signals = [
    'pan_verified', 'kyc_verified', 'mobile_verified', 'aadhaar_verified',
    'identity_match_status', 'identity_match_score', 'name_match_score'
]

leaked_signals = [s for s in identity_signals if s in X_train_processed.columns]
print(f"Identity/KYC signals tested : {len(identity_signals)}")
print(f"Identity signals in ML matrix: {leaked_signals}")

assert len(leaked_signals) == 0, f"Architectural Violation: Identity signals {leaked_signals} found in ML vector!"
print("\nIdentity Risk vs Credit Risk Separation: CERTIFIED & COMPLIANT")


Identity/KYC signals tested : 7
Identity signals in ML matrix: []

Identity Risk vs Credit Risk Separation: CERTIFIED & COMPLIANT


---
## SECTION 11 — Automated Sanity Checks & Validation Suite

We execute 12 automated validation assertions to certify the mathematical integrity, stability, and zero-leakage compliance of the Risk Scoring and Loan Decision Layer.

In [10]:
print("Executing Phase 3 Step 3 Risk Scoring & Decision Validation Suite...")

# 1. Estimated PD exists and is an array/Series
assert estimated_pd is not None and len(estimated_pd) > 0, "Gate 1 Failed: Estimated PD does not exist!"

# 2. Number of PD predictions equals number of test records (10)
assert len(estimated_pd) == len(X_test_processed) == 10, f"Gate 2 Failed: Expected 10 PD values, got {len(estimated_pd)}"

# 3. All PD values are strictly between 0 and 1
assert np.all((estimated_pd >= 0.0) & (estimated_pd <= 1.0)), "Gate 3 Failed: PD values outside [0, 1] range!"

# 4. No NaN PD values
assert not np.isnan(estimated_pd).any(), "Gate 4 Failed: NaN detected in estimated PD!"

# 5. No infinite PD values
assert not np.isinf(estimated_pd).any(), "Gate 5 Failed: Infinite values detected in estimated PD!"

# 6. Risk score exists and has 10 values
assert len(risk_scores) == 10, f"Gate 6 Failed: Expected 10 risk scores, got {len(risk_scores)}"

# 7. Risk score is between 0 and 100
assert np.all((risk_scores >= 0.0) & (risk_scores <= 100.0)), "Gate 7 Failed: Risk scores outside [0, 100] range!"

# 8. Risk tier contains ONLY {'LOW', 'MEDIUM', 'HIGH'}
assert set(risk_tiers).issubset({'LOW', 'MEDIUM', 'HIGH'}), f"Gate 8 Failed: Invalid risk tiers: {set(risk_tiers)}"

# 9. Decision contains ONLY {'APPROVE', 'MANUAL REVIEW', 'REJECT'}
assert set(loan_decisions).issubset({'APPROVE', 'MANUAL REVIEW', 'REJECT'}), f"Gate 9 Failed: Invalid decisions: {set(loan_decisions)}"

# 10. Final decision table has exactly one row per test applicant (10 rows)
assert len(decision_table) == 10, f"Gate 10 Failed: Decision table has {len(decision_table)} rows, expected 10!"

# 11. Actual default remains strictly separate from decision logic (verified via absence in decision input arguments)
assert 'Actual Default' in decision_table.columns and 'Decision' in decision_table.columns, "Gate 11 Failed: Columns missing"

# 12. Identity verification fields are 100% excluded from model feature matrix
assert all(s not in X_train_processed.columns for s in identity_signals), "Gate 12 Failed: Identity signals in feature matrix"

print("=" * 65)
print("PHASE 3 — STEP 3 RISK SCORING & DECISION: VALIDATION PASSED")
print("=" * 65)


Executing Phase 3 Step 3 Risk Scoring & Decision Validation Suite...
PHASE 3 — STEP 3 RISK SCORING & DECISION: VALIDATION PASSED


---
## SECTION 12 — Limitations & Production Roadmap Considerations

In enterprise data science, acknowledging dataset and modeling boundaries is critical for regulatory compliance and sound engineering:

1. **Synthetic Small-Sample Scope (50 Records)**:  
   The dataset contains 50 synthetic records (40 train / 10 test). While ideal for rapid prototyping and pipeline validation, statistical inferences cannot replace enterprise calibration on hundreds of thousands of historical borrower loan lifecycles.
2. **Uncalibrated Probabilities**:  
   The PD values used here are model-estimated probabilities $\hat{P}(y=1|X)$. In production banking, probabilities undergo formal calibration (e.g., Platt scaling, isotonic regression, beta calibration) to guarantee that an estimated PD of 10% corresponds to exactly 10 defaults per 100 loans in reality.
3. **Policy Threshold Assumptions**:  
   The cutoffs (`LOW_RISK_MAX_PD = 0.15`, `MEDIUM_RISK_MAX_PD = 0.40`) are demonstration policy parameters. In enterprise systems, thresholds are derived from optimization models balancing cost of capital, Expected Loss (EL), and profit margins.
4. **Demonstration Risk Score**:  
   The 0–100 score is a transparent internal demo metric, not an official credit bureau score.
5. **Production Governance Requirements**:  
   Production deployment requires continuous data drift monitoring, fairness and bias audits (DPDP Act 2023 compliance), explainability tooling (SHAP in Phase 4), and underwriting policy overrides.

---
## SECTION 13 — Phase 3 Summary & System Status

### 🔄 Complete End-to-End Decision Pipeline Architecture:

```
                      Customer Loan Application
                                  │
                                  ▼
                  Identity & KYC Verification Layer
              (PAN, Aadhaar KYC, Mobile OTP, Levenshtein Match)
                                  │
                                  ▼
                  Eligibility Gate (Identity Risk)
                     [FAIL] ──► Halted / Fraud Review
                     [PASS]
                                  │
                                  ▼
                      Credit Risk Feature Vector
               (14 Raw Features: Bureau, Income, DTI, Banking)
                                  │
                                  ▼
                      Phase 2 Preprocessing Pipeline
                   (StandardScaler + OneHotEncoder -> 16 Cols)
                                  │
                                  ▼
                      Phase 3 ML Scoring Engine
                 (Logistic Regression / Ensembles)
                                  │
                                  ▼
                      Estimated Probability of Default (PD)
                             (0.0 <= PD <= 1.0)
                                  │
                                  ▼
                       Demonstration Risk Score
                     (Risk Score = (1 - PD) * 100)
                                  │
                                  ▼
                         Risk Tier Assignment
                   (LOW: PD<0.15 | MEDIUM: 0.15-0.40 | HIGH: PD>=0.40)
                                  │
                                  ▼
                        Operational Decision Policy
                   ┌──────────────┬──────────────────┬──────────────┐
                   │              │                  │              │
                   ▼              ▼                  ▼              ▼
                APPROVE     MANUAL REVIEW          REJECT        HALT (KYC)
               (Low Risk)   (Medium Risk)       (High Risk)    (Identity Fail)
```

### 📋 Phase 3 Milestones Completion Status:

| Milestone | Description | Status |
|---|---|:---:|
| **Phase 3 — Step 1** | Baseline Logistic Regression Model (`notebooks/03_model_development.ipynb`) | ✅ **COMPLETE** |
| **Phase 3 — Step 2** | Tree-Based Models & Benchmark Comparison (`notebooks/04_tree_based_models.ipynb`) | ✅ **COMPLETE** |
| **Phase 3 — Step 3** | Risk Scoring & Loan Decision Layer (`notebooks/05_risk_scoring_and_decision.ipynb`) | ✅ **COMPLETE** |
| **Phase 3 Overall** | **Credit Risk ML Model Development & Decision Intelligence** | ✅ **COMPLETE** |
| **Phase 4** | **Model Evaluation & Explainability (SHAP, PR-AUC, Calibration)** | ⏳ **NOT STARTED** |

---

```
============================================================
  PHASE 3 — STEP 1: COMPLETE
  PHASE 3 — STEP 2: COMPLETE
  PHASE 3 — STEP 3: COMPLETE
  PHASE 3 OVERALL: COMPLETE
  PHASE 4: NOT STARTED
============================================================
```